# Rally E4B RP Merged Upload

Single-pass CPU bake of the A100/B75 RP checkpoint from `rally-e4b-sft-jun14v10`, then optional HF upload when `HF_TOKEN` is set.

In [ ]:
import os, platform, shutil
from pathlib import Path

print('python_platform=', platform.platform())
print('working_disk_free_gb=', round(shutil.disk_usage('/kaggle/working').free / 1024**3, 2))
print('input_dirs=', [str(p) for p in Path('/kaggle/input').glob('*')])


In [ ]:
import os, subprocess, sys, time

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'])
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    'huggingface_hub[cli]>=1.5.0', 'hf_transfer>=0.1.9', 'safetensors>=0.4.5',
    'git+https://github.com/huggingface/transformers.git',
])

secret_token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN') or ''
for attempt in range(5):
    if secret_token:
        break
    try:
        from kaggle_secrets import UserSecretsClient
        secret_token = UserSecretsClient().get_secret('HF_TOKEN')
        break
    except Exception as exc:
        print('hf_secret_attempt_failed=', attempt + 1, type(exc).__name__)
        time.sleep(10)
if secret_token:
    os.environ.setdefault('HF_TOKEN', secret_token)
    os.environ.setdefault('HUGGING_FACE_HUB_TOKEN', secret_token)
print('hf_secret_loaded=', bool(secret_token))


In [ ]:
import json, os, shutil, subprocess, sys
from pathlib import Path

REPO_URL = os.environ.get('HERETIC_TO_ONNX_REPO', 'https://github.com/alkahest-ai/heretic-to-onnx.git')
REPO_REF = os.environ.get('HERETIC_TO_ONNX_REF', 'main')
REPO_DIR = Path('/kaggle/working/heretic-to-onnx')
if REPO_DIR.exists():
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_REF])
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'checkout', REPO_REF])
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'])
else:
    subprocess.check_call(['git', 'clone', '--branch', REPO_REF, '--depth', '1', REPO_URL, str(REPO_DIR)])
sys.path.insert(0, str(REPO_DIR))
from scripts.kaggle_rally_artifacts import ensure_rp_merged, find_artifacts

artifact_name = os.environ.get('RALLY_TWO_STAGE_ARTIFACT_NAME', 'rally-e4b-two-stage-sft')
direct_id = os.environ.get('RALLY_HERETIC_MODEL_ID', 'coder3101/gemma-4-E4B-it-heretic')
artifacts = find_artifacts(os.environ.get('RALLY_ARTIFACT_DIR', ''), artifact_name)
merged_dir = Path(os.environ.get('RALLY_MERGED_OUTPUT_DIR', '/kaggle/working/a100-b75-merged'))
stage_b_scale = float(os.environ.get('RALLY_STAGE_B_SCALE', '0.75'))
if merged_dir.exists():
    shutil.rmtree(merged_dir, ignore_errors=True)
ensure_rp_merged(
    artifacts,
    base_model_id=direct_id,
    output_dir=merged_dir,
    stage_b_scale=stage_b_scale,
)

files = [{'name': path.name, 'size': path.stat().st_size} for path in sorted(merged_dir.iterdir()) if path.is_file()]
file_names = {item['name'] for item in files}
checkpoint_ok = 'model.safetensors' in file_names or 'model.safetensors.index.json' in file_names
required_ok = checkpoint_ok and 'config.json' in file_names and 'tokenizer_config.json' in file_names
repo_id = os.environ.get('RALLY_RP_MERGED_REPO', 'thomasjvu/rally-4b-rp-source-merged')
private = os.environ.get('RALLY_PRIVATE', '1') != '0'
upload = {'ok': False, 'skipped': True, 'reason': 'HF_TOKEN is not set'}
if os.environ.get('HF_TOKEN'):
    from huggingface_hub import HfApi
    api = HfApi(token=os.environ['HF_TOKEN'])
    api.create_repo(repo_id=repo_id, repo_type='model', private=private, exist_ok=True)
    commit = api.upload_folder(
        repo_id=repo_id,
        repo_type='model',
        folder_path=str(merged_dir),
        commit_message='Upload Rally E4B two-stage RP A100/B75 merged checkpoint from Kaggle',
    )
    upload = {'ok': True, 'skipped': False, 'commit_url': str(commit)}
report = {
    'ok': required_ok and (upload['ok'] or upload['skipped']),
    'repo_id': repo_id,
    'artifact_dir': str(artifacts),
    'stage_b_scale': stage_b_scale,
    'source_dir': str(merged_dir),
    'checkpoint_ok': checkpoint_ok,
    'required_ok': required_ok,
    'upload': upload,
    'files': files,
}
report_path = Path('/kaggle/working/rally-e4b-rp-merged-upload-report.json')
report_path.write_text(json.dumps(report, indent=2, sort_keys=True) + '\n')
print(json.dumps(report, indent=2, sort_keys=True))
